# 📊 E-Commerce Data Cleaning & Exploratory Data Analysis (EDA)
**Project**: Global E-Commerce & Customer 360 Analytics  
**Author**: Tushar Joshi | Data Analyst Portfolio  

---
### 📌 Notebook Objectives
1. Load raw e-commerce transaction data and perform initial data auditing.
2. Handle missing values, outliers, invalid order statuses, and data type casting.
3. Analyze key sales KPIs: Total Revenue, Average Order Value (AOV), Order Volume over time.
4. Explore regional performance, product category breakdown, and payment preferences.
5. Save the cleaned dataset for downstream RFM segmentation and cohort retention modeling.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as io

# Configure visualization defaults
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

print("Libraries imported successfully!")

## 1. Load Raw Data & Audit Schema

In [ ]:
# Load raw dataset
df_raw = pd.read_csv("../data/raw_ecommerce_data.csv")
print(f"Dataset Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
# Data info & missing value check
df_raw.info()
print("\n--- Missing Values Count ---")
print(df_raw.isnull().sum())

## 2. Data Wrangling & Feature Engineering

In [ ]:
df = df_raw.copy()

# Cast dates to datetime
df["order_date"] = pd.to_datetime(df["order_date"])

# Impute missing payment methods with Mode
mode_payment = df["payment_method"].mode()[0]
df["payment_method"] = df["payment_method"].fillna(mode_payment)

# Filter valid completed & returned orders for financial analysis
df_valid = df[df["order_status"] != "Cancelled"].copy()

# Temporal features
df_valid["year_month"] = df_valid["order_date"].dt.to_period("M").astype(str)
df_valid["order_year"] = df_valid["order_date"].dt.year
df_valid["day_name"] = df_valid["order_date"].dt.day_name()

print(f"Cleaned dataset records: {len(df_valid)} (Removed {len(df) - len(df_valid)} cancelled orders)")

## 3. Executive KPI Dashboard Summary

In [ ]:
total_revenue = df_valid["total_revenue"].sum()
total_orders = df_valid["order_id"].nunique()
total_customers = df_valid["customer_id"].nunique()
aov = df_valid["total_revenue"].mean()

print(f"💰 Total Revenue:        ${total_revenue:,.2f}")
print(f"📦 Total Completed Orders: {total_orders:,}")
print(f"👥 Unique Customers:      {total_customers:,}")
print(f"🛒 Average Order Value:   ${aov:.2f}")

## 4. Sales Trends & Seasonality Analysis

In [ ]:
monthly_trend = df_valid.groupby("year_month")["total_revenue"].sum().reset_index()

plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_trend, x="year_month", y="total_revenue", marker="o", color="#2563eb", linewidth=2.5)
plt.title("Monthly Revenue Trajectory (2024 - 2026)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Year-Month")
plt.ylabel("Revenue ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Regional & Product Category Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Category Revenue
cat_rev = df_valid.groupby("category")["total_revenue"].sum().sort_values(ascending=False).reset_index()
sns.barplot(data=cat_rev, x="total_revenue", y="category", palette="Blues_r", ax=axes[0])
axes[0].set_title("Revenue by Product Category", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Revenue ($)")

# Regional Distribution
reg_rev = df_valid.groupby("region")["total_revenue"].sum().sort_values(ascending=False).reset_index()
sns.barplot(data=reg_rev, x="region", y="total_revenue", palette="Greens_r", ax=axes[1])
axes[1].set_title("Revenue by Geographic Region", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Revenue ($)")

plt.tight_layout()
plt.show()

## 6. Key Takeaways & Export

In [ ]:
# Export clean data
os.makedirs("../data", exist_ok=True)
df_valid.to_csv("../data/cleaned_ecommerce_data.csv", index=False)
print("✅ EDA Complete. Cleaned data written to '../data/cleaned_ecommerce_data.csv'")